In [1]:
%load_ext autoreload
%autoreload 2
# 1. 导入所有依赖库
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3" 
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import numpy as np
import random

# 导入我们的核心模块
from models.apm_former import APM_Former_ImageOnly
from utils.dataset import get_adni_dataloaders
# 修正导入：只导入config里存在的变量
from utils.config import TRAIN_CSV, VAL_CSV, BATCH_SIZE, NUM_WORKERS, TRAIN_IMG_SIZE
from utils.logging_utils import get_logger
from torch.optim.lr_scheduler import SequentialLR, LinearLR, CosineAnnealingLR

from models.build_model import build_selected_model

import torch.nn.functional as F
from sklearn.metrics import confusion_matrix, accuracy_score, recall_score, precision_score, f1_score, roc_auc_score,roc_curve


# 日志
logger = get_logger("Train")

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    # torch.cuda.manual_seed_all(seed)  # 如果用多卡
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False



# seed 42 71.95% 74.39%
# seed 3407 71.95% 75.61%
# seed 1234 73.17% 69.51%
# seed 98 71.95% 73.17%
# seed 20  69.51%
# seed 1024


In [2]:
# 2. 超参数配置
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-3
NUM_EPOCHS = 80
GRADIENT_ACCUMULATION_STEPS = 4
NUM_CLASSES = 2
FEATURE_SIZE = 24
GUIDE_CHANNELS = 18
SEED = 1024
MODEL_TYPE = "APM_Former"
SAVE_PATH = f"checkpoints/best_model2_{MODEL_TYPE}.pth"


set_seed(SEED)

logger.info("=" * 50)
logger.info("🌟 本次实验配置档案 (Experiment Config) 🌟")
logger.info("=" * 50)
logger.info(f"▶ 模型类型 (MODEL_TYPE): {MODEL_TYPE}")
logger.info(f"▶ 随机种子 (SEED): {SEED}")
logger.info(f"▶ 峰值学习率 (LR): {LEARNING_RATE}")
# logger.info(f"▶ 预热轮数 (Warmup Epochs): {UNFREEZE_EPOCH}")
logger.info(f"▶ 批次大小 (Batch Size): {BATCH_SIZE}")
logger.info(f"▶ 优化器: AdamW (Weight Decay: {WEIGHT_DECAY})")
logger.info(f"▶ 架构说明: 启用浅层特征多尺度融合 (shallow_downsample)")
logger.info(f"▶ 架构说明: 启用 m_k 调制掩码")
logger.info("=" * 50)

logger.info(f"训练设备: {DEVICE}")
logger.info(f"批次大小: {BATCH_SIZE}")
logger.info(f"总轮数: {NUM_EPOCHS}")

2026-07-04 02:01:54,255 - Train - INFO - ==================================================
2026-07-04 02:01:54,256 - Train - INFO - 🌟 本次实验配置档案 (Experiment Config) 🌟
2026-07-04 02:01:54,257 - Train - INFO - ==================================================
2026-07-04 02:01:54,258 - Train - INFO - ▶ 模型类型 (MODEL_TYPE): APM_Former
2026-07-04 02:01:54,259 - Train - INFO - ▶ 随机种子 (SEED): 1024
2026-07-04 02:01:54,259 - Train - INFO - ▶ 峰值学习率 (LR): 0.0003
2026-07-04 02:01:54,260 - Train - INFO - ▶ 批次大小 (Batch Size): 2
2026-07-04 02:01:54,261 - Train - INFO - ▶ 优化器: AdamW (Weight Decay: 0.001)
2026-07-04 02:01:54,262 - Train - INFO - ▶ 架构说明: 启用浅层特征多尺度融合 (shallow_downsample)
2026-07-04 02:01:54,262 - Train - INFO - ▶ 架构说明: 启用 m_k 调制掩码
2026-07-04 02:01:54,263 - Train - INFO - ==================================================
2026-07-04 02:01:54,264 - Train - INFO - 训练设备: cuda
2026-07-04 02:01:54,264 - Train - INFO - 批次大小: 2
2026-07-04 02:01:54,265 - Train - INFO - 总轮数: 80


In [3]:
# 3. 加载训练集 + 验证集 DataLoader
logger.info("正在加载数据集...")
train_loader, val_loader = get_adni_dataloaders(
    train_csv=TRAIN_CSV,
    val_csv=VAL_CSV,
    batch_size=BATCH_SIZE,
    target_size=TRAIN_IMG_SIZE,
    num_workers=NUM_WORKERS,
    use_sampler=False
)

2026-07-04 02:01:54,301 - Train - INFO - 正在加载数据集...
2026-07-04 02:01:54,309 - utils.dataset - INFO - ✅ 成功加载 82 个样本
2026-07-04 02:01:54,309 - utils.dataset - INFO - 🧪 启用【原生数据分布 (Shuffle) + 基础增强】策略
2026-07-04 02:01:54,338 - utils.dataset - INFO - ✅ 成功加载 656 个样本
2026-07-04 02:01:54,339 - utils.dataset - INFO - 训练集批次: 328，验证集批次: 41


In [4]:
# 4. 初始化模型（完整版！）
logger.info("初始化 APM-Former 模型...")
model, UNFREEZE_EPOCH = build_selected_model(MODEL_TYPE, DEVICE, TRAIN_IMG_SIZE)

# 打印模型参数量
total_params = sum(p.numel() for p in model.parameters())
logger.info(f"模型总参数量: {total_params / 1e6:.2f} M")

2026-07-04 02:01:54,381 - Train - INFO - 初始化 APM-Former 模型...
2026-07-04 02:01:54,382 - Train - INFO - ============================================================
2026-07-04 02:01:54,383 - Train - INFO - 🚀 当前正在初始化的模型架构: APM_Former
2026-07-04 02:01:54,384 - Train - INFO - ============================================================
/home/ubuntu/Code/APM_Former_Project/models/apm_former.py:31: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowli

加载SwinUNETR预训练权重：checkpoints/model_swinvit.pt


2026-07-04 02:01:55,812 - Train - INFO - 📦 模型 [APM_Former] 总参数量: 15.80 M
2026-07-04 02:01:55,814 - Train - INFO - 模型总参数量: 15.80 M


In [5]:
# 5. 损失函数 + 优化器设置
from monai.losses import FocalLoss
from torch.optim.lr_scheduler import CosineAnnealingLR
import torch.optim as optim

# ==========================================
# 1. 损失函数 (保持不变)
# ==========================================
weights = torch.tensor([1.0, 1.2]).to(DEVICE)
criterion = FocalLoss(weight=weights, gamma=2.0, to_onehot_y=True).to(DEVICE)
# weights = torch.tensor([1.0, 4.0]).to(DEVICE)
# criterion = nn.CrossEntropyLoss(weight=weights).to(DEVICE)

# ==========================================
# 2. 冻结与优化器逻辑 (全兼容)
# ==========================================
if MODEL_TYPE == "APM_Former":
    logger.info("🔒 阶段一：冻结 Swin 主干网络，仅训练新增模块...")
    for name, param in model.named_parameters():
        if "swin_backbone" in name:
            param.requires_grad = False
        else:
            param.requires_grad = True
else:
    # 其他 Baseline 模型默认全员解冻，直接参与训练
    for param in model.parameters():
        param.requires_grad = True

# 统一定义优化器
optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()), 
    lr=3e-4, 
    weight_decay=WEIGHT_DECAY
)

# ==========================================
# 3. 🌟 修复 Bug：动态计算调度器的 T_max
# ==========================================
if UNFREEZE_EPOCH > 0:
    # 情形 A (APM_Former)：第一阶段的 T_max 是预热轮数 (比如 10)
    scheduler = CosineAnnealingLR(optimizer, T_max=UNFREEZE_EPOCH)
else:
    # 情形 B (ResNet 等 Baseline)：从头训到尾，T_max 直接是总轮数 (80)
    scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

2026-07-04 02:01:55,868 - Train - INFO - 🔒 阶段一：冻结 Swin 主干网络，仅训练新增模块...


In [6]:
# 对比实验：增加TV Loss
def total_variation_loss_3d(displacement_field):
    """
    计算 3D 形变场的 TV Loss，促使相邻像素的偏移方向和幅度保持平滑
    displacement_field shape: (Batch_size, 3, Depth, Height, Width)
    """
    # 计算三个空间维度的梯度（相邻体素的差值绝对值）
    d_diff = torch.abs(displacement_field[:, :, 1:, :, :] - displacement_field[:, :, :-1, :, :])
    h_diff = torch.abs(displacement_field[:, :, :, 1:, :] - displacement_field[:, :, :, :-1, :])
    w_diff = torch.abs(displacement_field[:, :, :, :, 1:] - displacement_field[:, :, :, :, :-1])
    
    # 将三个方向的平滑度误差求平均并相加
    tv_loss = torch.mean(d_diff) + torch.mean(h_diff) + torch.mean(w_diff)
    return tv_loss

In [7]:
# 6. 训练/验证函数
def train_epoch(model, loader, criterion, optimizer, device, accum_steps):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    # tv_weight = 1e-4  # 初始建议给一个较小的值，比如 0.001
    tv_weight = 1e-4

    optimizer.zero_grad()
    for idx, (images, labels) in enumerate(tqdm(loader, desc="训练中")):
        images = images.to(device)
        labels = labels.to(device)

        # 🌟 核心兼容性修改：先统统接住模型的输出，不着急解包 (unpack)
        outputs = model(images)
        
        # 🌟 智能分流：判断当前跑的是哪个模型
        if isinstance(outputs, tuple):
            # 情形 A: 这是你的 APM_Former (返回了4个值)
            logits = outputs[0]
            displacement_field = outputs[3]
            # cls_loss = criterion(logits, labels.unsqueeze(1))
            cls_loss = criterion(logits, labels.long())
            # 加入特有的 tv_loss
            tv_loss = total_variation_loss_3d(displacement_field)
            loss = cls_loss + tv_loss * tv_weight
        else:
            # 情形 B: 这是经典的 ResNet/ViT 等模型 (直接返回 logits，无形变场)
            logits = outputs
            loss = criterion(logits, labels.unsqueeze(1))

        # 梯度累加缩放
        loss = loss / accum_steps

        loss.backward()
        
        # 👇 梯度裁剪，防止爆炸
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        if (idx + 1) % accum_steps == 0:
            optimizer.step()
            optimizer.zero_grad()

        total_loss += loss.item() * accum_steps
        
        # 注意：这里的 preds 只是为了在进度条看个大概，真实的实力还要看 val_epoch 的动态阈值
        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(loader)
    acc = 100 * correct / total
    return avg_loss, acc


from sklearn.metrics import confusion_matrix

@torch.no_grad()
def val_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_labels = []
    all_probs = []  # 🌟 新增：专门用于收集预测为 pMCI 的概率，计算 AUC 必须用

    for images, labels in loader: 
        images = images.to(device)
        labels = labels.to(device)

        logits, _, _, _ = model(images)
        loss = criterion(logits, labels.unsqueeze(1)) # 如果用的是交叉熵，这里不用 unsqueeze

        total_loss += loss.item()
        
        # 获取预测类别 (0 或 1)
        # preds = torch.argmax(logits, dim=1)
        
        
        # 🌟 获取预测为类别 1 (pMCI) 的概率
        probs = F.softmax(logits, dim=1)[:, 1] 
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.detach().cpu().numpy())

    avg_loss = total_loss / len(loader)
    
    # ==========================
    # 🌟 计算所有医学黄金指标
    # ==========================
    # acc = accuracy_score(all_labels, all_preds) * 100
    # precision = precision_score(all_labels, all_preds, pos_label=1, zero_division=0) * 100
    # sen = recall_score(all_labels, all_preds, pos_label=1) * 100  # Sensitivity / Recall
    # spe = recall_score(all_labels, all_preds, pos_label=0) * 100  # Specificity (反向召回)
    # f1 = f1_score(all_labels, all_preds, pos_label=1,zero_division=0) * 100
    
    # 计算 AUC (注意：有些 batch 如果只有一个类别会报错，这里做了容错)
    try:
        auc = roc_auc_score(all_labels, all_probs) * 100
    except ValueError:
        auc = 0.0
    # 在计算完 auc 之后：
    fpr, tpr, thresholds = roc_curve(all_labels, all_probs)
    # Youden Index = TPR - FPR (即 Recall + Specificity - 1)
    optimal_idx = np.argmax(tpr - fpr)
    optimal_threshold = thresholds[optimal_idx]

    # 3. 🌟 使用最佳阈值重新生成预测标签 (替代掉原来死板的 argmax 或 >= 0.48)
    best_preds = [1 if p >= optimal_threshold else 0 for p in all_probs]

    # 4. 计算真实潜能指标
    adj_acc = accuracy_score(all_labels, best_preds) * 100
    adj_precision = precision_score(all_labels, best_preds, pos_label=1, zero_division=0) * 100
    adj_sen = recall_score(all_labels, best_preds, pos_label=1) * 100
    adj_spe = recall_score(all_labels, best_preds, pos_label=0) * 100
    adj_f1 = f1_score(all_labels, best_preds, pos_label=1, zero_division=0) * 100
    
    # 5. 生成对应的真实混淆矩阵
    cm = confusion_matrix(all_labels, best_preds, labels=[0, 1])
    
    logger.info(f"💡 [{MODEL_TYPE}] 动态最佳判定阈值: {optimal_threshold:.4f}")
    logger.info(f"👉 混淆矩阵: [类0: {cm[0][0]}, 误判1: {cm[0][1]}] | [漏判1: {cm[1][0]}, 类1: {cm[1][1]}]")
    logger.info(f"🏆 真实指标: ACC: {adj_acc:.2f}% | Precision: {adj_precision:.2f}% | SEN: {adj_sen:.2f}% | SPE: {adj_spe:.2f}% | F1: {adj_f1:.2f}% | AUC: {auc:.2f}%")
    return avg_loss, adj_acc, auc

In [8]:
# ==========================================
# 🌟 7. 终极主训练循环 (支持多模型 & 动态阈值 & 早停)
# ==========================================
best_val_acc = 0.0
patience =  30  # 早停容忍轮数
no_improve_epochs = 0

logger.info("=" * 60)
logger.info(f"开始训练模型: {MODEL_TYPE} | 优化目标: 最大化真实潜能 (adj_acc)")
logger.info("=" * 60)

for epoch in range(NUM_EPOCHS):
    
    # 👇 ================= 新增：智能解冻机关 ================= 👇
    # 只有 APM_Former 才需要预热和解冻，其他 Baseline 直接跳过
    if MODEL_TYPE == "APM_Former" and epoch == UNFREEZE_EPOCH:
        logger.info("\n" + "🚀" * 20)
        logger.info("🔓 阶段二触发：解冻 Swin 主干网络，开始全员微调！")
        logger.info("🚀" * 20)
        
        # 将所有参数解冻
        for param in model.parameters():
            param.requires_grad = True
            
        # 重新定义优化器（包含主干网络），并大幅降低学习率来保护老专家！
        # 这里用 1e-4 或 5e-5 进行微调最合适
        optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=WEIGHT_DECAY)
        
        # 重新定义剩下 epoch 的调度器
        scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS - UNFREEZE_EPOCH)
    
    logger.info(f"\nEpoch [{epoch+1}/{NUM_EPOCHS}] - Model: {MODEL_TYPE}")
    
    # 1. 训练一轮
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, DEVICE, GRADIENT_ACCUMULATION_STEPS)
    
    # 2. 验证一轮 (获取动态阈值算出的真实潜能 adj_acc 和 auc)
    val_loss, adj_acc, val_auc = val_epoch(model, val_loader, criterion, DEVICE)
    
    # 学习率衰减
    scheduler.step()

    # 打印本轮总结 (注意这里打印的是 adj_acc)
    logger.info(f"训练损失: {train_loss:.4f} | 训练准确率: {train_acc:.2f}%")
    logger.info(f"验证损失: {val_loss:.4f} | 验证准确率: {adj_acc:.2f}% | AUC: {val_auc:.2f}%")
    logger.info(f"当前学习率: {optimizer.param_groups[0]['lr']:.8f}")

    # 3. 基于真实潜能 adj_acc 保存最佳模型
    if adj_acc > best_val_acc:
        best_val_acc = adj_acc
        torch.save(model.state_dict(), SAVE_PATH)  # SAVE_PATH 里已经包含了具体的 MODEL_TYPE 名字
        no_improve_epochs = 0
        logger.info(f"✅ 最佳模型已保存至 {SAVE_PATH}！当前最高准确率: {best_val_acc:.2f}%")
    else:
        no_improve_epochs += 1
        
    # 4. 早停机制 (Early Stopping)
    if no_improve_epochs >= patience:
        logger.info(f"\n🛑 已经连续 {patience} 轮没有刷新最高纪录，触发早停机制！")
        logger.info(f"🛑 {MODEL_TYPE} 的训练提前结束，防止过拟合。")
        break

logger.info("\n🎉 所有训练流程彻底完成！")
logger.info(f"🏆 {MODEL_TYPE} 最终锁定的最佳真实验证准确率: {best_val_acc:.2f}%")

2026-07-04 02:01:56,030 - Train - INFO - ============================================================
2026-07-04 02:01:56,033 - Train - INFO - 开始训练模型: APM_Former | 优化目标: 最大化真实潜能 (adj_acc)
2026-07-04 02:01:56,035 - Train - INFO - ============================================================
2026-07-04 02:01:56,037 - Train - INFO - 
Epoch [1/80] - Model: APM_Former
训练中: 100%|██████████| 328/328 [01:14<00:00,  4.38it/s]
2026-07-04 02:03:21,617 - Train - INFO - 💡 [APM_Former] 动态最佳判定阈值: 0.4218
2026-07-04 02:03:21,618 - Train - INFO - 👉 混淆矩阵: [类0: 35, 误判1: 17] | [漏判1: 12, 类1: 18]
2026-07-04 02:03:21,619 - Train - INFO - 🏆 真实指标: ACC: 64.63% | Precision: 51.43% | SEN: 60.00% | SPE: 67.31% | F1: 55.38% | AUC: 62.24%
2026-07-04 02:03:21,621 - Train - INFO - 训练损失: 0.1906 | 训练准确率: 53.20%
2026-07-04 02:03:21,622 - Train - INFO - 验证损失: 0.1818 | 验证准确率: 64.63% | AUC: 62.24%
2026-07-04 02:03:21,623 - Train - INFO - 当前学习率: 0.00029266
2026-07-04 02:03:23,062 - Train - INFO - ✅ 最佳模型已保存至 checkpoints/best_mo